In [1]:
from pathlib import Path
import sys, importlib

sys.path.insert(0, str((Path.cwd() / "..").resolve()))

import _infra.nbtools as nbtools
nbtools = importlib.reload(nbtools)  # pick up the updated portable build helpers

# Shorthands
run               = nbtools.run
tools             = nbtools.tools
artifacts_dir     = nbtools.artifacts_dir
build_executable  = nbtools.build_executable
time_executable   = nbtools.time_executable

ART      = artifacts_dir()
LL_MAIN  = ART / "06_llvm_ir.ll"                          # produced by the first notebook
DRIVER_C = Path("../assets/drivers/driver_vecadd.c").resolve()
EXE_PATH = ART / "vecadd_runner"

print("Artifacts dir:", ART)
print("LLVM IR:", LL_MAIN)
print("Driver C:", DRIVER_C)
print("Using clang:", nbtools.clang_path())


Artifacts dir: /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/tutorial/artifacts
LLVM IR: /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/tutorial/artifacts/06_llvm_ir.ll
Driver C: /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/tutorial/assets/drivers/driver_vecadd.c
Using clang: /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/third-party/llvm/build/bin/clang-21


In [41]:
from pathlib import Path
import sys, importlib

# reload helpers (just in case)
sys.path.insert(0, str((Path.cwd() / "..").resolve()))
import _infra.nbtools as nbtools
nbtools = importlib.reload(nbtools)

ART      = nbtools.artifacts_dir()
LL_MAIN  = ART / "06_llvm_ir.ll"
DRIVER_C = Path("../assets/drivers/driver_vecadd.c").resolve()
EXE_PATH = ART / "vecadd_runner"

if not LL_MAIN.exists():
    raise FileNotFoundError(f"LLVM IR not found: {LL_MAIN}")

obj_ll = nbtools.compile_ll_to_obj(LL_MAIN, EXE_PATH.with_suffix(".ir.o"), opt_level="2")
obj_c  = nbtools.compile_c_to_obj(DRIVER_C, EXE_PATH.with_suffix(".drv.o"), opt_level="2")

print("LL obj:", obj_ll)
print("C  obj:", obj_c)
print("clang :", nbtools.clang_path())

LL obj: /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/tutorial/artifacts/vecadd_runner.ir.o
C  obj: /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/tutorial/artifacts/vecadd_runner.drv.o
clang : /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/third-party/llvm/build/bin/clang-21


In [42]:
import subprocess, shutil

objs = [EXE_PATH.with_suffix(".ir.o"), EXE_PATH.with_suffix(".drv.o")]
try:
    exe = nbtools.link_executable(objs, EXE_PATH, extra_ldflags=["-lm"])
    print("Built:", exe)
except subprocess.CalledProcessError as e:
    print("=== LINK COMMAND ===")
    print(" ".join(e.args if isinstance(e.args, list) else [str(a) for a in e.args]))
    print("\n=== LINKER STDERR ===\n")
    print(e.stderr)

    # Show symbols to diagnose common issues (e.g., missing _mlir_ciface_main)
    nm = shutil.which("nm") or "/usr/bin/nm"
    for o in objs:
        print(f"\n=== nm -gU {o} ===")
        try:
            out = subprocess.check_output([nm, "-gU", str(o)], text=True, stderr=subprocess.STDOUT)
            print(out)
        except Exception as nm_err:
            print("(nm failed)", nm_err)

    # Helpful hints based on common failures
    print("\n=== HINTS ===")
    print("- If you see 'undefined symbol: _mlir_ciface_main', your MLIR pipeline likely missed -emit-c-interface.")
    print("  Either add -emit-c-interface to the lowering, or change the driver 'extern' to the actual emitted symbol.")
    print("- If the symbols show something like '_main' or a mangled name, align the driver 'extern' to that.")
    print("- On macOS, SDK/sysroot flags are already applied by nbtools; no need to add them here.")
    raise

Built: /Users/admin/Programming/ESWEEK-tutorial/Cinnamon/tutorial/artifacts/vecadd_runner


In [43]:
# Block — run & profile
stats = time_executable(EXE_PATH, warmup=2, repeat=10)

print("Last stdout:\n", stats["last_stdout"], "\n")
print("Runs:", stats["runs"])
print("min   :", stats["min_s"], "s")
print("mean  :", stats["mean_s"], "s")
print("max   :", stats["max_s"], "s")
print("stdev :", stats["stdev_s"], "s")

Last stdout:
 C[0]   = 1.000000
C[1]   = 1.500000
C[123] = 62.500000
C[N-1] = 52428800.000000
Validation: OK 

Runs: 10
min   : 0.8716530660021817 s
mean  : 0.9213750098999298 s
max   : 1.0387079270003596 s
stdev : 0.05451865109968405 s
